# 采样分布审计：顺滑抓取是否意味着分布单一

## tl;dr
固定Lance版本334的3331回合中，93.0%曾满足原生成功条件；成功状态占9.95%帧，30.4%回合在首次成功前出现回落至桌面的过程。几乎所有回合都会抬起方块，首次抬起中位约0.85秒。动作覆盖的主要限制是旋转指令99.59%的平方幅度集中在yaw。数据描述不等于世界模型泛化能力验证。

## Context & Methods
分析单位为完整200帧回合。只读取既有数据，不启动仿真、训练或改变采样策略。

### Key Assumptions
- 保存的sim_state布局由代码及observation列交叉验证。原生成功按距离≤2.5cm和7个臂关节速度≤0.2rad/s重算，并与reward=1校验。
- 缺少contact impulses时不声称精确恢复native grasp。
- 每回合末行动作是官方collect的回绕值，状态—动作配对统计剔除它。
- 回落可能是主动释放，不能直接全部算作失败。
- 本数据点云实际在base坐标；物理状态使用world坐标。

In [1]:
from pathlib import Path
import importlib.util, json
import numpy as np
import pandas as pd
# If opened elsewhere, set PROJECT to this checkout.
PROJECT = Path.cwd().parents[1]
assert (PROJECT / 'scripts/analyze_sampling_distribution.py').is_file()
AUDIT = PROJECT / 'artifacts/distribution_audit_20260907'
DATASET = PROJECT / 'artifacts/wm_collection_full_20260907T044524Z/pickcube_expert_mixed.lance'
SNAPSHOT_VERSION = 334
print('Dataset:', DATASET.name, 'snapshot:', SNAPSHOT_VERSION)

Dataset: pickcube_expert_mixed.lance snapshot: 334


## Data
从固定快照重新执行同一份CPU分析代码，避免读到采样继续增长后的不同分母。结果输出到独立notebook_recomputed目录。

In [2]:
spec=importlib.util.spec_from_file_location('distribution_audit_module', PROJECT/'scripts/analyze_sampling_distribution.py')
module=importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
result=module.analyze(DATASET,AUDIT/'notebook_recomputed',snapshot_version=SNAPSHOT_VERSION)
reference=json.loads((AUDIT/'summary.json').read_text())
assert result['snapshot']['complete_episodes_analyzed']==reference['snapshot']['complete_episodes_analyzed']
assert result['episode_rates']==reference['episode_rates']
assert result['actions']==reference['actions']
print('Recomputed episode and action metrics exactly match the saved report.')

{
  "episodes": 3331,
  "states": 666200,
  "lance_version": 334,
  "native_success_episode_percent": 93.00510357250074,
  "final_success_percent": 0.24016811768237767,
  "table_return_before_success_percent": 30.411287901531068,
  "rotation_yaw_energy_percent": 99.59279894828796,
  "output": "/root/autodl-tmp/JEPA4R-4r-jepa-L3kEpP/4r-jepa-maniskill2-lewm/artifacts/distribution_audit_20260907/notebook_recomputed"
}
Recomputed episode and action metrics exactly match the saved report.


## Results
### 回合事件与帧占比
同一回合可以同时包含成功、释放、回落和再次升起，因此这些事件不是互斥类别。

In [3]:
episode_table=pd.DataFrame([{
 'metric':name,'count':item['count'],'episodes':item['denominator'],'percent':round(item['percent'],2)
} for name,item in result['episode_rates'].items()])
print(episode_table.to_string(index=False))
print('\nSuccess-frame percentage:',round(result['frame_rates']['recomputed_native_success']['percent'],3))

                               metric  count  episodes  percent
        any_recomputed_native_success   3098      3331    93.01
           final_frame_native_success      8      3331     0.24
                         ever_at_goal   3179      3331    95.44
                    ever_elevated_5cm   3330      3331    99.97
         ever_elevated_while_tcp_near   3330      3331    99.97
                   never_elevated_5cm      1      3331     0.03
          returns_to_table_after_lift   3323      3331    99.76
returns_to_table_before_first_success   1013      3331    30.41
        multiple_elevation_excursions   3291      3331    98.80

Success-frame percentage: 9.954


### 动作范围与完整性
旋转转回物理轴并应用控制器的向量范数限制后计算平方幅度比例；局部角度分量非零不一定代表独立roll/pitch探索。

In [4]:
print('Rotation energy XYZ fractions:', result['actions']['world_aligned_rotation_energy_fraction_xyz'])
print('State-layout checks:',result['integrity']['state_layout_cross_checks'])
assert result['integrity']['duplicate_episode_step_keys']==0
assert all(v['nonfinite_values']==0 for v in result['integrity']['fields'].values())
assert result['integrity']['state_layout_cross_checks']['success_predicate_vs_reward1_disagreements']==0
print('Finite numeric fields, unique episode-step keys, and success reconstruction cross-check passed.')

Rotation energy XYZ fractions: [0.0028781662695109844, 0.0011937855742871761, 0.9959279894828796]
State-layout checks: {'qpos_vs_observation_max_abs': 0.0, 'qvel_vs_observation_max_abs': 0.0, 'cube_vs_column_max_abs': 0.0, 'goal_distance_vs_stored_max_abs': 5.960464477539063e-08, 'success_predicate_vs_reward1_disagreements': 0}
Finite numeric fields, unique episode-step keys, and success reconstruction cross-check passed.


### 点云可见性抽查
独立抽32回合、6400帧，不把这部分样本量混作全部3331回合；红色阈值与真实方块位置联合校验。

In [5]:
visual=json.loads((AUDIT/'visual-summary.json').read_text())
print(json.dumps(visual,ensure_ascii=False,indent=2)[:8000])

{
  "snapshot": {
    "source_dataset": "/root/autodl-tmp/JEPA4R-4r-jepa-L3kEpP/4r-jepa-maniskill2-lewm/artifacts/wm_collection_full_20260907T044524Z/pickcube_expert_mixed.lance",
    "lance_version": 334,
    "numeric_snapshot_utc": "2026-09-07T07:12:52.623954+00:00",
    "audit_utc": "2026-09-07T07:19:36.817279+00:00",
    "sample_seed": 20260907,
    "sampling": "uniform random without replacement over complete snapshot episode IDs",
    "population_episodes": 3331,
    "episode_ids": [
      432,
      456,
      672,
      780,
      783,
      993,
      1042,
      1218,
      1278,
      1287,
      1396,
      1433,
      1471,
      1501,
      1561,
      1630,
      1679,
      1695,
      1927,
      1969,
      2097,
      2098,
      2299,
      2342,
      2486,
      2544,
      2691,
      2871,
      2876,
      2948,
      3129,
      3135
    ],
    "episodes_analyzed": 32,
    "frames_analyzed": 6400,
    "columns_read": [
      "episode_idx",
      "step_idx",
  

## Takeaways
- 当前数据包含大量有效搬运状态、成功后的释放和不顺利过程；初始抬起确实高度可靠。
- 更重要的风险是行为策略分布偏窄，尤其缺少独立roll/pitch和明显视觉缺失场景。
- 不应为了降低成功率故意制造无用失败；应针对实际部署需要的缺口设计额外探索并保留成功轨迹。
- 原训练代码随机划分窗口，重叠窗口不能证明未见轨迹泛化；可另做整回合隔离的预测与冻结PPO评估。
- 世界模型尚未完成训练，本报告不声称表征质量已经验证。